In [ ]:
from openai import OpenAI
import json

In [ ]:
from recipeprep.config import get_config
config = get_config()

## RAG

In [ ]:
# Dependencies are managed in pyproject.toml.

In [ ]:
# Set OPENAI_API_KEY in your environment before running this notebook.

In [ ]:
from recipeprep.generation import get_API_response, get_generate_sys_prompt, get_recipe
from recipeprep.retrieval import (
    build_nutrient_retriever,
    build_recipe_retriever,
    get_recipe_by_id,
    load_file_content,
    retrieve_food_and_nutrients,
    retrieve_similar_recipe_id,
)

In [ ]:
# Retrieval dependencies are managed inside recipeprep.retrieval.

## Retrive the nutrients from nutrient map

In [ ]:
# Nutrient metadata is prepared by recipeprep.retrieval.

In [ ]:
file_path = config.nutrient_map_path
retriever = build_nutrient_retriever(file_path)

In [ ]:
# The persistent nutrient vector store is loaded above.

In [ ]:
# Vector-store creation is handled by build_nutrient_retriever().

In [ ]:
# retriever is ready for nutrient queries.

In [ ]:
# Nutrient lookup is imported from recipeprep.retrieval.

## Retrive similar recipe


In [ ]:
# TODO: change to remove processed_output
file_path_final_recipes = config.datasets_dir / "Archive/filtered_12_recipes_init_128.json"

with open(file_path_final_recipes, "r") as file:
    data = json.load(file)

for record in data:
    record["pure_ingredients"] =  ", ".join(sorted(record.get("pure_ingredients", [])))

file_path_final_recipes = config.datasets_dir / "Archive/test_filtered_12_recipes_init_128.json"
with open(file_path_final_recipes, "w") as file:
    json.dump(data, file)

In [ ]:
# Legacy metadata experiment retained for reference only.
def legacy_metadata_func_recipe(record: dict, metadata: dict) -> dict:

  metadata["recipe_title"] = record.get("recipe_title")

  metadata["pure_ingredients"] =   ", ".join(sorted(record.get("pure_ingredients", [])))
  metadata["recipe_id"]  = record.get("recipe_id")
  return metadata


In [ ]:
file_path_final_recipes = config.datasets_dir / "Archive/filtered_12_recipes_init_128.json"

with open(file_path_final_recipes, "r") as file:
    data = json.load(file)

for record in data:
    record["pure_ingredients"] =  ", ".join(sorted(record.get("processed_output", {}).get("pure_ingredients", [])))

file_path_final_recipes = config.datasets_dir / "Archive/test_filtered_12_recipes_init_128.json"
with open(file_path_final_recipes, "w") as file:
    json.dump(data, file)

In [ ]:
def legacy_processed_output_metadata(record: dict, metadata: dict) -> dict:

  metadata["recipe_title"] = record.get("recipe_title")

  metadata["pure_ingredients"] =   ", ".join(sorted(record.get("processed_output", {}).get("pure_ingredients", [])))
  metadata["recipe_id"]  = record.get("recipe_id")
  return metadata


In [ ]:
retriever_recipe = build_recipe_retriever(file_path_final_recipes)

In [ ]:
# The persistent recipe vector store is loaded above.

In [ ]:
# Similar-recipe lookup is imported from recipeprep.retrieval.

In [ ]:
input_ingre_list = ['shrimp','onions','garlic']
input_ingre_list = sorted(ingredient.lower() for ingredient in input_ingre_list)
# best_match = find_closest_recipes(retriever_recipe, input_ingre_list)
recipe_id = retrieve_similar_recipe_id(retriever_recipe,input_ingre_list)


In [ ]:
# Recipe lookup by ID is imported from recipeprep.retrieval.

## Calculate Health Score

In [ ]:
def get_health_score_with_rag(client,retriever,temp,topp,recipe):
  recipe_data = json.loads(recipe)
  # Extract title, ingredients, and instructions safely
  recipe_title = recipe_data["title"]
  recipe_ingredient_list = recipe_data["ingredients"]
  pure_ingredients = recipe_data["pure_ingredients"]
  instructions = recipe_data["instructions"]

  nutrient_map=[]
  for ingredient in pure_ingredients:
    matched_food, nutrients=retrieve_food_and_nutrients(retriever,ingredient)
    nutrient_map.append(nutrients)

  if not nutrient_map:
    return{
        "error": "No relevant nutrient map found for the given ingredient name."
    }
  sys_prompt=""
  user_prompt=f"""
  You are a helpful assistant that can evaluate the recipes' healthiness.
  You only need to consider 7 key macronutrients and their ranges to assess a recipe’s healthiness:
  Proteins: 10%-15% of total energy
  Carbohydrates: 55%-75% of total energy
  Sugars: less than 10% of total energy
  Sodium: less than 5 grams
  Fats: 15%-30% of total energy
  Saturated Fats: less than 10% of total energy
  Fibers: more than 25 grams

  Here's the generated recipe:
  Recipe Title: {recipe_title}
  Ingredients and Measurements: {recipe_ingredient_list}
  Nutrient Map: {nutrient_map}

  Evaluation Instructions:
  - Find the 7 key macronutrients of each ingredient (ingredient_name in the nutrient map). Add up each types of macronutrient of each ingreident to get the total content of each macronutrient.
  - Evaluate if each macronutrient is in range of evaluation criteria (1 point if yes, 0 if no).
  - Use one sentence to explain why each macronutrients is 1 or 0.
  - Sum the points to get a health score (0-7).

  The sample evaluation result and health score for the first recipe:
  Summary of Points:
  Proteins: 0 points
  Carbohydrates: 1 point (potatoes provide substantial carbohydrates)
  Sugars: 1 point (natural sugars in potatoes and cream are likely to be less than 10% of total energy)
  Sodium: 1 point (assuming moderate salt use, likely to stay under 5 grams)
  Fats: 1 point (butter, olive oil, cream are high in fats, possibly within the 15%-30% range)
  Saturated Fats: 0 points (butter, cream, and sour cream likely push this over 10%)
  Fibers: 0 points (likely less than 25 grams, as potatoes are not high in fibers)
  Total Health Score: 4

  Calculate the health score. The output should only contain the following attributes:
  - title: recipe title
  - ingredients: each ingredient must have measurement in the recipe
  - instructions: step by step instructions with numbering
  - summary of points: name of the key macronutrients and their corresponding points. (use one sentence to explain why each macronutrients is 1 or 0 like the sample above, right after the point, don't make another paragraph)
  - total health score: the number of the total health score.
  The output must be a string in JSON format that contains the above attributes. Do not need to specify the format type(i.e. json) at the beginning of the output string.
  """
  response=get_API_response(client,user_prompt,user_prompt,temp,topp)
  return response

## Recipe Generation

In [ ]:
client = OpenAI()  # Reads OPENAI_API_KEY from the environment
file_path_recipe = config.datasets_dir / "Archive/filtered_12_recipes_init_128.json"
recipe_generated_path = "./generated_result/recipe_generated.json"

In [ ]:
# TODO: change to remove processed_out
def legacy_get_generate_sys_prompt_new(recipes_file_path, ingredients,retriever_ingre,retriever_recipe,provide_example=True):

  ingredients = sorted([ingredient.lower() for ingredient in ingredients])

  sample_recipes_text=''
  if provide_example:
    all_sample_reciples = load_file_content(recipes_file_path)
    #Retrive sample recipe
    recipe_examples = []
    similar_recipes_ids = retrieve_similar_recipe_id(retriever_recipe,ingredients)

    for eachID in similar_recipes_ids:
      eachRecipe = get_recipe_by_id(all_sample_reciples, eachID)
      title = eachRecipe.get("recipe_title", "Untitled Recipe")
      ingredients = eachRecipe.get("ingredients", [])
      pure_ingredients = eachRecipe.get("pure_ingredients", [])
      instructions = eachRecipe.get("step_by_step_instructions", [])
      nutrients_points = eachRecipe.get("summary_of_points", {})
      health_score = eachRecipe.get("total_health_score", 0)

      formatted_recipe = (
          f"Title: {title}\n"
          f"Ingredients: {', '.join(ingredients)}\n"
          f"Pure Ingredients: {' '.join(pure_ingredients)}\n"
          f"Instructions: {' '.join(instructions)}\n"
          f"Summary of Points: {' '.join(nutrients_points)}\n"
          f"Health Score: {health_score}\n"
      )
      recipe_examples.append(formatted_recipe)
    sample_recipes_text = "\n\n".join(recipe_examples)



  #Retrive nutrient content
  nutrient_map=[]
  for ingredient in ingredients:
    matched_food, nutrients=retrieve_food_and_nutrients(retriever_ingre,ingredient)
    nutrient_map.append(nutrients)


  print(sample_recipes_text)
  print(nutrient_map)

  sys_prompt = f'''
    You are a helpful assistant that can generate a healthy recipe based on some sample recipes and their evaluations of healthiness.

    Here's some sample recipes and nutrient map for reference:
    {f'sample recipes: {sample_recipes_text}' if provide_example else ''}
    nutrient map: {nutrient_map}

    Here's the evaluation criteria:
    You only need to consider 7 key macronutrients and their ranges to assess a recipe's healthiness:
    Proteins: 10%-15% of total energy
    Carbohydrates: 55%-75% of total energy
    Sugars: less than 10% of total energy
    Sodium: less than 2.5 grams
    Fats: 15%-30% of total energy
    Saturated Fats: less than 10% of total energy
    Fibers: more than 12.5 grams

    Evaluation Instructions:
    - Find the 7 key macronutrients of each ingredient (ingredient_name in the nutrient map). Add up each types of macronutrient of each ingreident to get the total content of each macronutrient.
    - Evaluate if each macronutrient is in range of evaluation criteria (1 point if yes, 0 if no).
    - Use one sentence to explain why each macronutrients is 1 or 0.
    - Sum the points to get a health score (0-7).

    Task:
    - Create a recipe using only user-provided ingredients and tools that has the highest health score (health score must be greater or equal to 5).
    - Check which macronutrients do not satisfy and get 0 point.
    - Then try to choose a propriate ingredients with proper measurements in the list of ingredients. For example, if portein is marked as 0, then use the protein in the ingredients in the generated recipe or use a proper portion
      of that protein to satisfy the requirements of protein content. You can increase or decrease the amount of ingredients to satisfy the requirements.
      Another example, if the amount of broccoli is not enough, simply increase the measreument of broccoli in the ingredients.
      If there is no ingredients available to satisfy the specific requirements, then provide suggestions on what ingredients to purchase.
    - Calculate the summary of points and health score again of the improved recipe based on evaluation instructions. Use the adjustments in step 2 to calculate the points and score, don't use the original generated recipe.
    - If the calculated health score is less than 5, repeat the steps.

    The output must have the following attributes:
    - title: recipe title
    - ingredients: each ingredient must have measurement in the recipe
    - instructions: step by step instructions with numbering
    - summary of points: name of the key macronutrients and their corresponding points. (use one sentence to explain why each macronutrients is 1 or 0, right after the point, don't make another paragraph)
    - total health score: the number of the total health score.
    - suggestions: tell user what seasonings are used in this generated recipe with measurements. If there is suggestion on what to purchase, also mention that.

    The output must be a string in JSON format that contains the above attributes. Do not need to specify the format type(i.e. json) at the beginning of the output string.
  '''
  return sys_prompt

In [ ]:
def legacy_get_generate_sys_prompt(recipes_file_path, ingredients,tools, time, retriever_ingre,retriever_recipe,provide_example=True):

  ingredients = sorted([ingredient.lower() for ingredient in ingredients])
  user_preferred_cooking_time = time

  sample_recipes_text=''
  if provide_example:
    all_sample_reciples = load_file_content(recipes_file_path)
    #Retrive sample recipe
    recipe_examples = []
    similar_recipes_ids = retrieve_similar_recipe_id(retriever_recipe,ingredients)

    for eachID in similar_recipes_ids:
      eachRecipe = get_recipe_by_id(all_sample_reciples, eachID)
      title = eachRecipe.get("recipe_title", "Untitled Recipe")
      ingredients = eachRecipe.get("ingredients", [])
      pure_ingredients = eachRecipe.get("processed_output", {}).get("pure_ingredients", [])
      instructions = eachRecipe.get("processed_output", {}).get("step_by_step_instructions", [])
      nutrients_points = eachRecipe.get("summary_of_points", {})
      cooking_time = eachRecipe.get("processed_output", {}).get("cooking_time", {})
      required_tools = eachRecipe.get("processed_output", {}).get("required_tools", [])
      health_score = eachRecipe.get("total_health_score", 0)

      formatted_nutrients_points = "\n".join([f"{nutrient}: {value}" for nutrient, value in nutrients_points.items()])
      formatted_cooking_time = "\n".join([f"{key}: {value}" for key, value in cooking_time.items()])

      formatted_recipe = (
          f"Title: {title}\n"
          f"Ingredients: {', '.join(ingredients)}\n"
          f"Pure Ingredients: {', '.join(pure_ingredients)}\n"
          f"Instructions: {' '.join(instructions)}\n"
          f"Summary of Points: \n{(formatted_nutrients_points)}\n"
          f"Cooking Time: \n{(formatted_cooking_time)}\n"
          f"Required Tools: {', '.join(required_tools)}\n"
          f"Health Score: {health_score}\n"
      )
      recipe_examples.append(formatted_recipe)
    sample_recipes_text = "\n\n".join(recipe_examples)

    #print(sample_recipes_text)

  #Retrive nutrient content
  nutrient_map=[]
  for ingredient in ingredients:
    matched_food, nutrients=retrieve_food_and_nutrients(retriever_ingre,ingredient)
    nutrient_map.append(nutrients)


  sys_prompt = f'''
    You are a helpful assistant that can generate a balanced and healthy recipe based on some sample recipes and their evaluations of healthiness.

    Here's some sample recipes and nutrient map for reference:
    {f'sample recipes: {sample_recipes_text}' if provide_example else ''}
    nutrient map: {nutrient_map}

    User preferred cooking time: {user_preferred_cooking_time}

    Here's the evaluation criteria:
    You only need to consider 7 key macronutrients and their ranges to assess a recipe's healthiness:
    Proteins: 10%-15% of total energy
    Carbohydrates: 55%-75% of total energy
    Sugars: less than 10% of total energy
    Sodium: less than 2.5 grams
    Fats: 15%-30% of total energy
    Saturated Fats: less than 10% of total energy
    Fibers: more than 12.5 grams

    Evaluation Instructions:
    - Find the 7 key macronutrients of each ingredient (ingredient_name in the nutrient map). Add up each types of macronutrient of each ingreident to get the total content of each macronutrient.
    - Evaluate if each macronutrient is in range of evaluation criteria (1 point if yes, 0 if no).
    - Use one sentence to explain why each macronutrients is 1 or 0.
    - Sum the points to get a health score (0-7).

    Task:
    - Create a recipe using only user-provided ingredients and tools that has the highest health score (health score must be greater or equal to 5).
    - The cooking time is the sum of each step in Cooking time. Calculate the cooking time of the generated recipe. The generated recipe has to be close to the user prefeeded cooking time.
    - Check which macronutrients that do not satisfy and get 0 point.
    - Then try to choose a propriate ingredients with proper measurements in the list of ingredients. For example, if portein is marked as 0, then use the protein in the ingredients in the generated recipe or use a proper portion
      of that protein to satisfy the requirements of protein content. You can increase or decrease the amount of ingredients to satisfy the requirements.
      Another example, if the amount of broccoli is not enough, simply increase the measreument of broccoli in the ingredients.
      If there is no ingredients available to satisfy the specific requirements, then provide suggestions on what ingredients to purchase.
    - Calculate the summary of points and health score again of the improved recipe based on evaluation instructions. Use the adjustments in step 2 to calculate the points and score, don't use the original generated recipe.
    - If the calculated health score is less than 5, repeat previous steps.

    The output must have the following attributes:
    - title: recipe title
    - ingredients: each ingredient must have measurement in the recipe (each ingredient starts with a new line, don't use \n)
    - pure_ingredients: ingredients without measurement (each ingredient starts with a new line, don't use \n)
    - instructions: step by step instructions with numbering (each instruction starts with a new line, don't use \n)
    - required_tools: tools that will be required to cook this recipe
    - cooking_time: total cooking time
    - suggestions: tell user what seasonings are used in this generated recipe with measurements.

    Additional Formatting Instructions of output:
    - Each list item (ingredients, pure ingredients, instructions, and required tools) must appear on a new line in the output without the use of \n characters. The newlines should be part of the string in the final output.
    - The final output must be a well-formed JSON string without specifying the format type (e.g., json) at the beginning. The output should only contain the structured JSON data as per the attributes listed.

  '''
  return sys_prompt

In [ ]:
# JSON loading is imported from recipeprep.retrieval.

In [ ]:
def legacy_get_recipe(client, ingredients, tools, time, temp, topp, file_path,provide_example=True):

    user_prompt = (
        f"I have the following ingredients: {', '.join(ingredients)}.\n"
        f"I also have these cooking requirements: {', '.join(tools)}.\n"
        f"I prefer the cooking time to be within: {(time)} minutes.\n"
    )
    sys_prompt = get_generate_sys_prompt(file_path, ingredients,tools, time, retriever,retriever_recipe,provide_example)
    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temp,
            top_p=topp
        )

        recipe_generated = completion.choices[0].message.content.strip()

        return recipe_generated
    except Exception as e:
        return f"An error occurred: {e}"

#### Testing: Generated Recipe with provided example

In [ ]:
# main
print("Welcome to RecipePrep!")

ingredients_input = input("Enter your ingredients: ").strip().split(",")
avail_ingredients = [ingredient.strip() for ingredient in ingredients_input]

tools_input = input("Enter your cooking requirements: ").strip().split(",")

avail_tools = [tool.strip() for tool in tools_input]

time_input = input("Enter your preferred cooking time (in min): ").strip().split(",")

# Generate recipe using the inputs
print("\nGenerating your recipe...\n")
recipe = get_recipe(
    client,
    avail_ingredients,
    avail_tools,
    time_input,
    1,
    1,
    file_path_recipe,
    retriever_nutrient=retriever,
    retriever_recipe=retriever_recipe,
)
print(recipe)


#### Testing: Generated example Without providing example


In [ ]:
client_2 = OpenAI()  # Reads OPENAI_API_KEY from the environment

In [ ]:
# main
print("Welcome to RecipePrep!")

ingredients_input = input("Enter your ingredients, separated by commas: ").strip().split(",")
avail_ingredients = [ingredient.strip() for ingredient in ingredients_input]

tools_input = input("Enter your cooking tools, separated by commas: ").strip().split(",")
avail_tools = [tool.strip() for tool in tools_input]

# Generate recipe using the inputs
print("\nGenerating your recipe...\n")
recipe = get_recipe(
    client_2,
    avail_ingredients,
    avail_tools,
    30,
    1,
    1,
    file_path_recipe,
    retriever_nutrient=retriever,
    retriever_recipe=retriever_recipe,
    provide_example=False,
)
eval = get_health_score_with_rag(client, retriever, 1, 1, recipe)

with open("final_recipe_with_score", "w") as file:
    json.dump(eval, file, indent=4)
print("Recipe generated and saved")

print("Here's your recipe:\n")
print(eval)